In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from maite_datasets.image_classification import MilitaryVehicles

from dataeval import Metadata
from dataeval.config import set_seed
from dataeval.data import Filter, Limit, Operation, Relabel, Shuffle, View

set_seed(0)  # For reproducibility

In [3]:
vehicles = MilitaryVehicles("./data", image_set="test", download=True, lazy=True)
index2label = vehicles.metadata["index2label"]
class_index = {name: index for index, name in index2label.items()}
families = MilitaryVehicles.hierarchy["vehicle"]["military"]["land vehicle"]  # family -> its vehicle types
labels = Metadata(vehicles).class_labels  # the class of each image, to see what a view kept

print("source dataset size:", len(vehicles))
print("source vocabulary:", index2label)
print("families:", {family: len(types) for family, types in families.items()})

source dataset size: 1621
source vocabulary: {0: '2S19 MSTA', 1: '30N6E', 2: 'BM-30', 3: 'BMD', 4: 'BMP-1', 5: 'BMP-2', 6: 'BMP-T15', 7: 'BRDM', 8: 'BTR-60', 9: 'BTR-70', 10: 'BTR-80', 11: 'D-30', 12: 'Iskander', 13: 'MT LB', 14: 'Pantsir-S1', 15: 'Rs-24', 16: 'T-14', 17: 'T-62', 18: 'T-64', 19: 'T-72', 20: 'T-80', 21: 'T-90', 22: 'TOS-1', 23: 'Tornado'}
families: {'Tank': 6, 'BMP': 3, 'BTR': 4, 'Self Propelled Artillery': 5, 'Air Defense': 4, 'MT LB': 1, 'BMD': 1}


In [4]:
tank_classes = [class_index[name] for name in families["Tank"]]
view = View(vehicles, operations=[Filter(classes=tank_classes), Limit(200)])

print("view size:", len(view))
print("source is unchanged:", len(vehicles))
print("first 10 source indices behind the view:", view.resolve_indices()[:10])

view size: 200
source is unchanged: 1621
first 10 source indices behind the view: [1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1252, 1253]


In [5]:
# TEST ASSERTION CELL ###
assert len(view) == 200
assert len(vehicles) == 1621
assert len(index2label) == 24
assert {family: len(types) for family, types in families.items()}["Tank"] == 6

In [6]:
limit_then_filter = View(vehicles, operations=[Limit(200), Filter(classes=tank_classes)])
filter_then_limit = View(vehicles, operations=[Filter(classes=tank_classes), Limit(200)])

print("Limit(200) -> Filter(Tank):", len(limit_then_filter), "(tanks *within* the first 200 images)")
print("Filter(Tank) -> Limit(200):", len(filter_then_limit), "(the first 200 tanks)")

Limit(200) -> Filter(Tank): 0 (tanks *within* the first 200 images)
Filter(Tank) -> Limit(200): 200 (the first 200 tanks)


In [7]:
# TEST ASSERTION CELL ###
assert np.all(np.diff(labels) >= 0)  # the split is stored class by class
assert len(limit_then_filter) == 0
assert len(filter_then_limit) == 200

In [8]:
first_500 = View(vehicles, operations=[Limit(500)])
sample_500 = View(vehicles, operations=[Shuffle(seed=0), Limit(500)])

print("classes in Limit(500):", len(set(labels[first_500.resolve_indices()])), "of", len(index2label))
print("classes in Shuffle -> Limit(500):", len(set(labels[sample_500.resolve_indices()])), "of", len(index2label))

classes in Limit(500): 8 of 24
classes in Shuffle -> Limit(500): 24 of 24


In [9]:
# TEST ASSERTION CELL ###
assert len(set(labels[first_500.resolve_indices()])) == 8
assert len(set(labels[sample_500.resolve_indices()])) == 24
assert len(sample_500) == 500

In [10]:
family_names = tuple(families)
family_of = {vehicle_type: family for family, types in families.items() for vehicle_type in types}


def source_types(v: View) -> list[str]:
    """The original vehicle types sitting behind a view's items."""
    return sorted({index2label[int(labels[i])] for i in v.resolve_indices()})


relabel_first = View(vehicles, operations=[Relabel(family_of, family_names), Filter(classes=[1])])
filter_first = View(vehicles, operations=[Filter(classes=[1]), Relabel(family_of, family_names)])

print("vocabulary after Relabel:", relabel_first.metadata.get("index2label"))
print()
print("Relabel -> Filter(classes=[1]):", len(relabel_first), "images, source types", source_types(relabel_first))
print("Filter(classes=[1]) -> Relabel:", len(filter_first), "images, source types", source_types(filter_first))

vocabulary after Relabel: {0: 'Tank', 1: 'BMP', 2: 'BTR', 3: 'Self Propelled Artillery', 4: 'Air Defense', 5: 'MT LB', 6: 'BMD'}

Relabel -> Filter(classes=[1]): 176 images, source types ['BMP-1', 'BMP-2', 'BMP-T15']
Filter(classes=[1]) -> Relabel: 21 images, source types ['30N6E']


In [11]:
# TEST ASSERTION CELL ###
assert relabel_first.metadata.get("index2label") == dict(enumerate(family_names))
assert family_names[1] == "BMP"
assert index2label[1] == "30N6E"
assert source_types(relabel_first) == ["BMP-1", "BMP-2", "BMP-T15"]
assert source_types(filter_first) == ["30N6E"]
assert (len(relabel_first), len(filter_first)) == (176, 21)
assert int(np.argmax(filter_first[0][1])) == family_names.index("Air Defense")  # relabeled to its family

In [12]:
class KeepEvery(Operation):
    """Cardinality: keep every nth item of the current selection."""

    def __init__(self, n: int) -> None:
        self.n = n

    def apply(self, view: View) -> None:
        view.selection = view.selection[:: self.n]


class Dim(Operation):
    """Content: darken every image by a factor, as in dusk imagery. Registers a transform, reads no data here."""

    def __init__(self, factor: float) -> None:
        self.factor = factor

    def apply(self, view: View) -> None:
        view.map(lambda datum: ((np.asarray(datum[0]) * self.factor).astype(np.uint8), datum[1], datum[2]))


custom = View(vehicles, operations=[Shuffle(seed=0), Limit(100), KeepEvery(10), Dim(0.5)])
first = custom.resolve_indices()[0]

print("custom view size:", len(custom))
print("source indices:", custom.resolve_indices())
print(
    "mean pixel value of the first item, source:",
    round(float(np.asarray(vehicles[first][0]).mean()), 1),
    "view:",
    round(float(np.asarray(custom[0][0]).mean()), 1),
)

custom view size: 10
source indices: [1229, 382, 486, 1538, 524, 77, 90, 753, 1101, 1112]


mean pixel value of the first item, source: 170.8 view: 85.1


In [13]:
# TEST ASSERTION CELL ###
assert custom.resolve_indices() == View(vehicles, [Shuffle(seed=0), Limit(100)]).resolve_indices()[::10]
assert len(custom) == 10
assert np.array_equal(np.asarray(custom[0][0]), (np.asarray(vehicles[first][0]) * 0.5).astype(np.uint8))

In [14]:
nested = View(View(vehicles, [Filter(classes=tank_classes)]), [Limit(25)])

print("operations, innermost first:", nested.operation_groups)
print("original dataset:", type(nested.root).__name__)
print("source indices behind the first 5 items:", nested.resolve_indices()[:5])

operations, innermost first: [[Filter(where=None, classes=[16, 17, 18, 19, 20, 21], drop='both')], [Limit(size=25)]]
original dataset: MilitaryVehicles
source indices behind the first 5 items: [0, 1, 2, 3, 4]


In [15]:
# TEST ASSERTION CELL ###
assert len(nested.operation_groups) == 2
assert nested.root is vehicles